In [1]:
import logging
import sqlite3
from langchain_community.llms import HuggingFacePipeline
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from transformers import pipeline


logging.basicConfig(
    filename='integration_log.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger()

logger.info("Notebook started: LangChain + HuggingFace + SQLite integration.")
print("✅ Logging setup complete. Logs will be saved to integration_log.log")



✅ Logging setup complete. Logs will be saved to integration_log.log


In [3]:
logger.info("Creating in-memory SQLite database and inserting sample data...")
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE facts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    topic TEXT,
    content TEXT
)
""")

sample_data = [
    ("AI", "Artificial intelligence is the simulation of human intelligence processes by machines."),
    ("ML", "Machine learning is a subset of AI focused on building systems that learn from data."),
    ("NLP", "Natural language processing enables computers to understand, interpret, and respond to human language.")
]

cursor.executemany("INSERT INTO facts (topic, content) VALUES (?, ?)", sample_data)
conn.commit()

logger.info("Database created and populated with sample data.")
print("✅ Database setup complete and populated with example records.")


✅ Database setup complete and populated with example records.


In [5]:
logger.info("Loading data from database for embedding...")
cursor.execute("SELECT content FROM facts")
rows = cursor.fetchall()
texts = [r[0] for r in rows]

print("✅ Loaded data from database:")
for text in texts:
    print("-", text)


✅ Loaded data from database:
- Artificial intelligence is the simulation of human intelligence processes by machines.
- Machine learning is a subset of AI focused on building systems that learn from data.
- Natural language processing enables computers to understand, interpret, and respond to human language.


In [7]:
logger.info("Initializing Hugging Face embedding model...")

# Using a compact embedding model for demonstration
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

logger.info("Creating FAISS vector store...")
vectorstore = FAISS.from_texts(texts, embedding=embeddings)

print("✅ Vector store created using Hugging Face embeddings.")


C:\Users\vpyas\AppData\Local\Temp\ipykernel_27172\31875571.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\vpyas\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vpyas\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector store created using Hugging Face embeddings.


In [9]:
logger.info("Loading Hugging Face language model...")
generator = pipeline("text-generation", model="distilgpt2", max_new_tokens=100)
llm = HuggingFacePipeline(pipeline=generator)

print("✅ LLM pipeline (DistilGPT2) loaded successfully.")


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

C:\Users\vpyas\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vpyas\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better per

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


✅ LLM pipeline (DistilGPT2) loaded successfully.


C:\Users\vpyas\AppData\Local\Temp\ipykernel_27172\2161631386.py:3: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=generator)


In [11]:
logger.info("Setting up RetrievalQA chain...")

prompt = PromptTemplate(
    template="Answer the question using the context below.\n\nContext: {context}\n\nQuestion: {question}",
    input_variables=["context", "question"]
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

print("✅ RetrievalQA chain created and ready.")


✅ RetrievalQA chain created and ready.


In [13]:
query = "What is machine learning?"
logger.info(f"Running query: {query}")

result = qa.run(query)

print("🔹 Query:", query)
print("🔹 Answer:", result)

logger.info(f"Query result: {result}")


C:\Users\vpyas\AppData\Local\Temp\ipykernel_27172\1844645658.py:4: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa.run(query)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


🔹 Query: What is machine learning?
🔹 Answer: Answer the question using the context below.

Context: Machine learning is a subset of AI focused on building systems that learn from data.

Artificial intelligence is the simulation of human intelligence processes by machines.

Natural language processing enables computers to understand, interpret, and respond to human language.

Question: What is machine learning?
Machine learning is a subset of AI focused on building systems that learn from data.
Sophisticated computational computing is an important part of this process.
The computer algorithm learns and learns from, or learns from, a human.
Machine learning is a subset of AI focused on building systems that learn from data.
Machine learning is a subset of AI focused on building systems that learn from data.
Machine learning is a subset of AI focused on building systems that learn from data.



In [ ]:
print("Last 10 lines of log:")
with open("integration_log.log", "r") as log_file:
    lines = log_file.readlines()[-10:]
    for line in lines:
        print(line.strip())
